# Amazon Bedrock AgentCore Runtime での分散マルチエージェントソリューション

## 概要

このチュートリアルでは、各エージェントを独自の Bedrock AgentCore Runtime に独立してホストし、異なるエージェントフレームワークで構築する方法を学習します。その後、分散マルチエージェントソリューションのためにそれらの間の通信を有効にします。

この例では、以下を作成します：
1. プログラミングや技術的なトラブルシューティングに関する技術的な質問に答えることに特化した技術エージェント（`tech_agent`）
2. 会社の福利厚生に特化した HR エージェント（`hr_agent`）
3. 質問を技術エージェントまたは HR エージェントにルーティングするオーケストレーターエージェント（`orchestrator_agent`）

これら3つのエージェントを組み合わせることで、ユーザーの質問を適切なサブエージェントにルーティングできるスーパーバイザーを持つマルチエージェント構成が得られます。このシステムは、従業員が会社で持つ可能性のある幅広い質問に答えることができます。


### チュートリアルの詳細


| 情報         | 詳細                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| チュートリアルタイプ       | 会話型                                                                   |
| エージェントタイプ          | マルチエージェント（ツールとしてエージェントを呼び出すスーパーバイザー）                                                                           |
| エージェントフレームワーク   | Strands Agents & LangGraph                                                                  |
| LLM モデル           | Anthropic Claude Haiku 4.5                                                        |
| チュートリアルコンポーネント | AgentCore Runtime でエージェントをホストし、マルチエージェントコラボレーションを有効化 |
| チュートリアル垂直領域   | クロス垂直                                                                   |
| 例の複雑さ  | 中程度                                                                             |
| 使用するSDK            | Amazon BedrockAgentCore Python SDK と boto3                                     |

### チュートリアルアーキテクチャ

このチュートリアルでは、3つのエージェントを Bedrock AgentCore runtime にデプロイする方法について説明します。オーケストレーターには Strands Agent、Tech エージェントには Strands Agent、HR エージェントには LangGraph エージェントを使用します。シンプルなエージェントを使用して、エージェントフレームワークの組み合わせでマルチエージェントシステムを構成し、各エージェントを独自の AgentCore Runtime にデプロイする方法を示します。

![alt text](./architecture.png)


### チュートリアルの主な機能

* Amazon Bedrock AgentCore Runtime で複数のエージェントをホスト
* 各エージェントが独立してホストされるマルチエージェントソリューションの作成


## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Python 3.10+
* AWS 認証情報
* Amazon Bedrock AgentCore SDK
* Strands Agents
* LangGraph

In [ ]:
!uv add -r requirements.txt --active

In [1]:
import os

# Set an environment variable
# os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = "ap-northeast-1"

## エージェントの作成

まず、各エージェントに対して3つの個別の IAM ロールを作成します。これにより、他のエージェントとは独立して、各エージェントに最小権限のアクセス許可を定義できます。

In [2]:
from utils import create_agentcore_role

tech_agent_name="tech_agent"
tech_agent_iam_role = create_agentcore_role(agent_name=tech_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
tech_agent_role_arn = tech_agent_iam_role['Role']['Arn']
tech_agent_role_name = tech_agent_iam_role['Role']['RoleName']
print(tech_agent_role_arn)
print(tech_agent_role_name)

hr_agent_name="hr_agent"
hr_agent_iam_role = create_agentcore_role(agent_name=hr_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
hr_agent_role_arn = hr_agent_iam_role['Role']['Arn']
hr_agent_role_name = hr_agent_iam_role['Role']['RoleName']
print(hr_agent_role_arn)
print(hr_agent_role_name)

orchestrator_agent_name="orchestrator_agent"
orchestrator_iam_role = create_agentcore_role(agent_name=orchestrator_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
orchestrator_role_arn = orchestrator_iam_role['Role']['Arn']
orchestrator_role_name = orchestrator_iam_role['Role']['RoleName']
print(orchestrator_role_arn)
print(orchestrator_role_name)

attaching role policy agentcore-tech_agent-role
arn:aws:iam::195049633937:role/agentcore-tech_agent-role
agentcore-tech_agent-role
attaching role policy agentcore-hr_agent-role
arn:aws:iam::195049633937:role/agentcore-hr_agent-role
agentcore-hr_agent-role
attaching role policy agentcore-orchestrator_agent-role
arn:aws:iam::195049633937:role/agentcore-orchestrator_agent-role
agentcore-orchestrator_agent-role


### ヘルパー関数：

* `configure_runtime` ヘルパー関数は、各エージェントのランタイム設定をセットアップするために使用されます。この例では、スターターツールキットを使用して、エントリーポイント、作成した実行ロール、requirements ファイルで AgentCore Runtime デプロイを設定します。また、スターターキットを設定して、起動時に Amazon ECR リポジトリを自動作成します。
* `check_status` ヘルパー関数は、AWS アカウントにデプロイされた各ランタイムをチェックして、作成が成功し、エージェントが使用可能であることを検証するために使用されます。

In [3]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import time


def configure_runtime(agent_name, agentcore_iam_role, python_file_name):
    boto_session = Session(region_name=os.getenv("AWS_DEFAULT_REGION"))
    region = boto_session.region_name

    agentcore_runtime = Runtime()

    response = agentcore_runtime.configure(
        entrypoint=python_file_name,
        execution_role=agentcore_iam_role['Role']['Arn'],
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name=agent_name
    )
    return response, agentcore_runtime

def check_status(agent_runtime):
    status_response = agent_runtime.status()
    status = status_response.endpoint['status']
    end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
    while status not in end_status:
        time.sleep(10)
        status_response = agent_runtime.status()
        status = status_response.endpoint['status']
        print(status)
    return status

In [4]:
# set the current working directory to be the tech_agent folder
import os
os.chdir('./tech_agent')
print(os.getcwd())

/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent


### テクニカルサポートエージェントの作成（Strands Agents）

Strands と Amazon Bedrock モデルを使用してテクニカルサポートエージェントから始めましょう。次のセルを実行すると、`./tech_agent` ディレクトリにエージェント固有のロジックを含む `tech_agent.py` ファイルが作成されます。

アプリは `BedrockAgentCoreApp()` で定義され、呼び出し関数 `strands_agent_bedrock` は `@app.entrypoint` デコレータで装飾され、`app.run()` コマンドがファイルの最後にあることに注意してください。

In [5]:
%%writefile tech_agent.py

from strands import Agent, tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    system_prompt="あなたは親切なテクニカルサポートアシスタントです。技術的なトラブルシューティングやプログラミングに関するユーザーからの質問に対応できます。"
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Writing tech_agent.py


#### エージェントの起動：

まず、`configure_runtime` ヘルパー関数を使用して、エージェントデプロイに必要な `.bedrock_agentcore.yaml`、`.dockerignore`、`Dockerfile` を作成します。次に、ランタイムで `.launch()` を呼び出して、イメージを ECR にプッシュし、AWS 環境に AgentCore Runtime を作成します。


In [6]:
_, tech_agent_runtime = configure_runtime("tech_agent", tech_agent_iam_role, "tech_agent.py")
tech_launch_result = tech_agent_runtime.launch()
tech_agent_id = tech_launch_result.agent_id
tech_agent_arn = tech_launch_result.agent_arn

print(tech_agent_arn)

Entrypoint parsed: file=/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/tech_agent.py, bedrock_agentcore_name=tech_agent
Configuring BedrockAgentCore agent: tech_agent
Generated .dockerignore
Generated Dockerfile: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/Dockerfile
Generated .dockerignore: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/.dockerignore
Setting 'tech_agent' as default agent
Bedrock AgentCore configured: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-m

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-tech_agent


Getting or creating CodeBuild execution role for agent: tech_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
✓ Role created: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-c1ae289f0f
Using .dockerignore with 44 patterns
Uploaded source to S3: tech_agent/source.zip
Created CodeBuild project: bedrock-agentcore-tech_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild mo

arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/tech_agent-uAy36F6CRS


#### テクニカルエージェントの ARN をパラメータストアに保存

これにより、シンプルなエージェントレジストリが作成され、テクニカルエージェントの AgentCore Runtime ARN を永続的に保存して検索できるようになります

In [7]:
import boto3
import time

ssm = boto3.client('ssm')
ssm.put_parameter(
    Name=f'/agents/tech_agent_arn',
    Value=tech_agent_arn,
    Type='String',
    Overwrite=True
)

{'Version': 1,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '56aa55a0-6b92-4c6d-8401-811875f7b5df',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Sun, 04 Jan 2026 06:31:39 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '56aa55a0-6b92-4c6d-8401-811875f7b5df',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

#### エージェントのテスト

エージェントをテストするために、まずテクニカルエージェントの AgentCore Runtime のステータスを確認し、使用可能であることを確認します。\
`tech_agent_runtime` で `.invoke()` を使用して、エージェントランタイムが設定され、期待どおりに動作していることを検証します。

In [8]:
status = check_status(tech_agent_runtime)
print(status)

Retrieved Bedrock AgentCore status for: tech_agent


READY


In [9]:
invoke_response = tech_agent_runtime.invoke({"prompt": "Macでウィンドウを最小化するショートカット、1文で"})
invoke_response

{'ResponseMetadata': {'RequestId': '4571a984-5b88-4dcc-8c44-8548869bd1e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 04 Jan 2026 06:32:06 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4571a984-5b88-4dcc-8c44-8548869bd1e8',
   'baggage': 'Self=1-695a0964-6d431d09294aa8ad3a37e1a8,session.id=4b44554e-d200-472c-8cb0-2d3da8a6c553',
   'x-amzn-bedrock-agentcore-runtime-session-id': '4b44554e-d200-472c-8cb0-2d3da8a6c553',
   'x-amzn-trace-id': 'Root=1-695a0964-07e817cb71507ac63d8a9de3;Parent=71d4b1770e2ad1a3;Sampled=1;Self=1-695a0964-6d431d09294aa8ad3a37e1a8'},
  'RetryAttempts': 0},
 'runtimeSessionId': '4b44554e-d200-472c-8cb0-2d3da8a6c553',
 'traceId': 'Root=1-695a0964-07e817cb71507ac63d8a9de3;Parent=71d4b1770e2ad1a3;Sampled=1;Self=1-695a0964-6d431d09294aa8ad3a37e1a8',
 'baggage': 'Self=1-695a0964-6d431d09294aa8ad3a37e1a8,session.id=4b44554e-d200-472c-8cb0-2d3da8a6c553',
 'contentType': 

### HR エージェントの作成（LangGraph Agents）

HR エージェントを作成するために、同様のプロセスに従います。ただし、今回は基盤となるエージェントロジックが LangGraph で構築されていることに気づくでしょう。エージェントフレームワークのこの変更は、AgentCore Runtime の設定方法に影響を与えません。次のセルを実行すると、`./hr_agent` ディレクトリに `hr_agent.py` ファイルが作成されます。

テクニカルサポートエージェントと同様に、アプリを `BedrockAgentCoreApp()` で定義し、呼び出し関数 `langgraph_bedrock` を `@app.entrypoint` デコレータで装飾し、`app.run()` コマンドをファイルの最後に配置します。

In [10]:
# set the current working directory to be the hr_agent folder
import os

os.chdir('../hr_agent')
print(os.getcwd())

/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent


In [12]:
%%writefile hr_agent.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

@tool
def get_vacation_info():
    """Get remaining vacation days balance for the current year"""  # Dummy implementation
    return "you have 12 days off remaining this year"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock
    
    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )
    
    # Bind tools to the LLM
    tools = [get_vacation_info]
    llm_with_tools = llm.bind_tools(tools)
    
    # System message
    system_message = f"""あなたは親切な人事サポートアシスタントです。休暇や福利厚生に関するユーザーの質問に回答できます。
主な会社福利厚生は以下の通りです
- 従業員100%、扶養家族75%の保険料をカバーする包括的な健康保険
- 柔軟な有給休暇制度（年間20日＋病気休暇5日）
- 401(k)退職金制度（会社6%マッチング、即時権利確定）
- 月額100ドルの健康手当（ジム会員費やフィットネス活動に利用可）

その他人事関連のお問い合わせは、1-800-ASKHRまでお電話ください。"""
    
    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # Create the graph
    graph_builder = StateGraph(MessagesState)
    
    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # Set entry point
    graph_builder.set_entry_point("chatbot")
    
    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    
    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

Overwriting hr_agent.py


#### エージェントの起動：

再度、`configure_runtime` ヘルパー関数を使用して、エージェントデプロイに必要な `.bedrock_agentcore.yaml`、`.dockerignore`、`Dockerfile` を作成します。次に、HR エージェントランタイムで `.launch()` を呼び出して、イメージを ECR にプッシュし、AWS 環境に AgentCore Runtime を作成します。

In [13]:
_, hr_agentcore_runtime = configure_runtime("hr_agent", hr_agent_iam_role, "hr_agent.py")
hr_launch_result = hr_agentcore_runtime.launch()
hr_agent_id = hr_launch_result.agent_id
hr_agent_arn = hr_launch_result.agent_arn

print(hr_agent_arn)

Entrypoint parsed: file=/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/hr_agent.py, bedrock_agentcore_name=hr_agent
Configuring BedrockAgentCore agent: hr_agent
Generated .dockerignore
Generated Dockerfile: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/Dockerfile
Generated .dockerignore: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/.dockerignore
Setting 'hr_agent' as default agent
Bedrock AgentCore configured: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-hr_agent


Getting or creating CodeBuild execution role for agent: hr_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
✓ Role created: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-1a1b25b3e4
Using .dockerignore with 44 patterns
Uploaded source to S3: hr_agent/source.zip
Created CodeBuild project: bedrock-agentcore-hr_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitori

arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:runtime/hr_agent-N62XFxCDC5


#### HR エージェントの ARN をパラメータストアに保存

これにより、シンプルなエージェントレジストリが継続され、HR エージェントの AgentCore Runtime ARN を永続的に保存して検索できるようになります

In [14]:
import boto3
import time

ssm = boto3.client('ssm')
ssm.put_parameter(
    Name=f'/agents/hr_agent_arn',
    Value=hr_agent_arn,
    Type='String',
    Overwrite=True  
)

{'Version': 1,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '2679c127-e770-4799-a7f7-a13c739c92a0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Sun, 04 Jan 2026 06:34:07 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '2679c127-e770-4799-a7f7-a13c739c92a0',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

#### エージェントのテスト

HR エージェントの AgentCore Runtime のステータスを確認し、使用可能であることを確認しましょう。\
`hr_agentcore_runtime` で `.invoke()` を使用して、AgentCore Runtime が設定され、期待どおりに動作していることを検証します。

In [15]:
status = check_status(hr_agentcore_runtime)
status

Retrieved Bedrock AgentCore status for: hr_agent


'READY'

In [ ]:
# Test your agent
invoke_response = hr_agentcore_runtime.invoke({"prompt": "残りの有給休暇は何日ですか？"})
invoke_response

{'ResponseMetadata': {'RequestId': '683adcb5-b59a-448c-8fad-13d01c163b23',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 04 Jan 2026 06:34:26 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '683adcb5-b59a-448c-8fad-13d01c163b23',
   'baggage': 'Self=1-695a09e3-3d0485b9027799e53d094b6d,session.id=9f20d0d1-08d6-4547-9f15-631c02cfe531',
   'x-amzn-bedrock-agentcore-runtime-session-id': '9f20d0d1-08d6-4547-9f15-631c02cfe531',
   'x-amzn-trace-id': 'Root=1-695a09e3-73cdac2a6a3a6ebf00b883fd;Parent=e3dc50f5ae9ad7b1;Sampled=1;Self=1-695a09e3-3d0485b9027799e53d094b6d'},
  'RetryAttempts': 0},
 'runtimeSessionId': '9f20d0d1-08d6-4547-9f15-631c02cfe531',
 'traceId': 'Root=1-695a09e3-73cdac2a6a3a6ebf00b883fd;Parent=e3dc50f5ae9ad7b1;Sampled=1;Self=1-695a09e3-3d0485b9027799e53d094b6d',
 'baggage': 'Self=1-695a09e3-3d0485b9027799e53d094b6d,session.id=9f20d0d1-08d6-4547-9f15-631c02cfe531',
 'contentType': 

### オーケストレーターエージェントの作成（Strands Agents）

3番目のエージェントであるオーケストレーターには、再度 Strands をエージェントフレームワークとして使用します。エージェントを作成する前に、以前に作成した AgentCore Runtime の実行ロールを更新して、テクニカルサポートエージェントと HR エージェントを呼び出すためのアクセス許可を付与する必要があります。

以下の `update_orchestrator_permissions` 関数は、サブエージェントの ARN とパラメータストアに登録されたエージェントの ARN を受け取り、オーケストレーターエージェントにサブエージェントの DEFAULT ランタイムエンドポイントを呼び出す権限を付与します。また、オーケストレーターエージェントにパラメータストアからエージェント ARN を取得する権限も付与します。

In [17]:
# Let's update the orchestrator agentcore exeuction role so it has permissions to invoke the required subagents
# the orchestrator also needs needs permissions to retrieve the sub agent arns from parameter store
import json 

# retrieve the runtime arn from parameter store
ssm = boto3.client('ssm')
response = ssm.get_parameter(Name='/agents/tech_agent_arn')
tech_agent_arn = response['Parameter']['Value']
tech_agent_parameter_arn = response['Parameter']['ARN']

ssm = boto3.client('ssm')
response = ssm.get_parameter(Name='/agents/hr_agent_arn')
hr_agent_arn = response['Parameter']['Value']
hr_agent_parameter_arn = response['Parameter']['ARN']

def update_orchestrator_permissions(sub_agent_arns: list, sub_agent_parameter_arns: list, orchestrator_name: str):
    iam_client = boto3.client('iam')
    orchestrator_permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:InvokeAgentRuntime"
                ],
                "Resource": [ sub_agent_arn + "/runtime-endpoint/DEFAULT" for sub_agent_arn in sub_agent_arns ] + [ sub_agent_arn for sub_agent_arn in sub_agent_arns ]
            },
            {
                "Effect": "Allow",
                "Action": [
                    "ssm:GetParameter"
                ],
                "Resource": [sub_agent_parameter_arn for sub_agent_parameter_arn in sub_agent_parameter_arns]

            }]
    }
        
    rsp = iam_client.put_role_policy(
        RoleName=orchestrator_name,
        PolicyName="subagent_permissions-new",
        PolicyDocument=json.dumps(orchestrator_permissions)
    )
    return rsp

rsp = update_orchestrator_permissions([tech_agent_arn, hr_agent_arn], [tech_agent_parameter_arn, hr_agent_parameter_arn], orchestrator_role_name)
print(rsp)

{'ResponseMetadata': {'RequestId': '3955ed6a-695b-48a6-8555-7ad6cbb84cc2', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 04 Jan 2026 06:35:35 GMT', 'x-amzn-requestid': '3955ed6a-695b-48a6-8555-7ad6cbb84cc2', 'content-type': 'text/xml', 'content-length': '206'}, 'RetryAttempts': 0}}


In [18]:
# set the current working directory to be the orchestrator_agent folder
import os
os.chdir('../orchestrator_agent')
print(os.getcwd())

/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent


以下のセルを実行すると、`./orchestrator_agent`ディレクトリにエージェント固有のロジックを含む`orchestrator_agent.py`ファイルが作成されます。オーケストレーターエージェントには2つのツールが利用可能です：1. `call_tech_agent`、2. `call_HR_agent`。これらのツールはいずれも、`./orchestrator_agent`フォルダ内に既に存在する`invoke_agent_utils.py`ファイルで事前定義されているinvoke_agent_utils関数を使用します。サブエージェントは、boto3経由で利用可能なinvoke_agent_runtimeアクションを使用してツールとして呼び出されます。

このより複雑なエージェント設定においても、Bedrock AgentCoreランタイムの設定は変わりません。アプリは`BedrockAgentCoreApp()`で定義され、呼び出し関数`strands_agent_bedrock_streaming`は`@app.entrypoint`デコレータで装飾され、`app.run()`コマンドはファイルの末尾にあることに注意してください。

In [19]:
%%writefile orchestrator_agent.py

import argparse
import json
import boto3
import logging

from strands import Agent, tool
from strands_tools import calculator 
from strands.models import BedrockModel

from bedrock_agentcore.runtime import BedrockAgentCoreApp

from invoke_agent_utils import invoke_agent_with_boto3

logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

def get_agent_arn(agent_name: str) -> str:
    """
    Retrieve agent ARN from Parameter Store
    """
    try:
        ssm = boto3.client('ssm')
        response = ssm.get_parameter(
            Name=f'/agents/{agent_name}_arn'
        )
        return response['Parameter']['Value']
    except Exception as err:
        print(err)
        raise err

@tool
def call_tech_agent(user_query):
    """ call the tech agent """ 
    # print("Calling tech agent")
    try:
        tech_agent_arn = get_agent_arn ("tech_agent")
        result = invoke_agent_with_boto3(tech_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.exception("Exception calling tech agent: ")
    return result

@tool
def call_HR_agent(user_query):
    """ Get the HR agent """ 
    print("Calling HR agent")
    try:
        hr_agent_arn = get_agent_arn("hr_agent")
        print(hr_agent_arn)
        result = invoke_agent_with_boto3(hr_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.error(f"Exception calling hr agent: {e}", exc_info=True)
    return result


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    system_prompt="あなたは有用なアシスタントです。あなたの役割はユーザーの質問を理解し、適切な専門エージェントに振り分けることです。技術部門や人事部門のエージェントに連絡を取るためのツールが用意されています。",
    tools=[call_tech_agent, call_HR_agent]
)

def parse_event(event):
    """
    エージェントからのストリーミングイベントを解析し、フォーマットされた出力を返す
    """
    # Skip events that don't need to be displayed
    if any(key in event for key in ['init_event_loop', 'start', 'start_event_loop']):
        return ""
    
    # Text chunks from supervisor
    if 'data' in event and isinstance(event['data'], str):
        return event['data'] 
    
    
    # Handle text messages from the assistant
    if 'event' in event:
        event_data = event['event']
        
        # Beginning of a tool use
        if 'contentBlockStart' in event_data and 'start' in event_data['contentBlockStart']:
            if 'toolUse' in event_data['contentBlockStart']['start']:
                tool_info = event_data['contentBlockStart']['start']['toolUse']
                return f"\n\n[Executing: {tool_info['name']}]\n\n"        

    return ""

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """ストリーミング機能を備えたエージェントを呼び出す

    この関数は、非同期ジェネレータを使用して AgentCore Runtime でストリーミング応答を実装する方法を示します
    """
    user_input = payload.get("prompt")
    #print("User input:", user_input)
    
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            text = parse_event(event)
            if text:  # Only return non-empty responses
                yield text
                
            #if "data" in event:
            #    yield event["data"]
            
    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response


if __name__ == "__main__":
    app.run()

Writing orchestrator_agent.py


#### エージェントの起動：

再度、`configure_runtime` ヘルパー関数を使用して、エージェントデプロイに必要な `.bedrock_agentcore.yaml`、`.dockerignore`、`Dockerfile` を作成します。次に、オーケストレーターエージェントランタイムで `.launch()` を呼び出して、イメージを ECR にプッシュし、AWS 環境に AgentCore Runtime を作成します。

In [20]:
_, orchestrator_agentcore_runtime = configure_runtime("orchestrator_agent", orchestrator_iam_role, "orchestrator_agent.py")
orchestrator_launch_result = orchestrator_agentcore_runtime.launch()

Entrypoint parsed: file=/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/orchestrator_agent.py, bedrock_agentcore_name=orchestrator_agent
Configuring BedrockAgentCore agent: orchestrator_agent
Generated .dockerignore
Generated Dockerfile: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/Dockerfile
Generated .dockerignore: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/.dockerignore
Setting 'orchestrator_agent' as default agent
Bedrock AgentCore configured: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-orchestrator_agent


Getting or creating CodeBuild execution role for agent: orchestrator_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
✓ Role created: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-00cafffea8
Using .dockerignore with 44 patterns
Uploaded source to S3: orchestrator_agent/source.zip
Created CodeBuild project: bedrock-agentcore-orchestrator_agent-builder
Starting CodeBuild build (this may take several minutes).

#### エージェントのテスト

オーケストレーターエージェントの AgentCore Runtime のステータスを確認し、使用可能であることを確認しましょう。

今回は、ユーティリティの `invoke_agent_with_boto3` 関数を使用してオーケストレーターエージェントをテストできます。テクニカルサポートエージェントと HR エージェントの両方の呼び出しをトリガーする質問をオーケストレーターに尋ねましょう。

In [21]:
status = check_status(orchestrator_agentcore_runtime)
print(status)

from invoke_agent_utils import invoke_agent_with_boto3


result = invoke_agent_with_boto3 (orchestrator_launch_result.agent_arn, "私の特典について教えてください。また、MacにBluetoothマウスを接続する方法も教えてください。")

Retrieved Bedrock AgentCore status for: orchestrator_agent


READY
Invoking agent...
Processing streaming response...

ご質問ありがとうございます。2つのご質問をいただいていますので、それぞれ適切なエージェントに振り分けさせていただきます。

[Executing: call_HR_agent]



[Executing: call_tech_agent]

お疲れ様です。2つのご質問についての回答がまいりました。

## 📋 **あなたの特典について**

当社では以下の福利厚生を提供しています：

- **🏥 健康保険**：従業員100%、扶養家族75%カバー
- **🏖️ 有給休暇**：年間20日 + 年間5日の病気休暇
- **💰 401(k)退職金制度**：会社が給与の6%をマッチング（即時権利確定）
- **💪 健康手当**：月額100ドル（ジム会費やフィットネス活動に利用可能）

現在の残り有給日数を確認されたい場合は、お知らせください。

---

## 🖱️ **MacにBluetoothマウスを接続する方法**

1. **マウスの電源をON**にしてペアリングモードに設定
2. **Apple メニュー → システム設定 → Bluetooth** を開く
3. 利用可能なデバイス一覧からマウスを選択
4. 「接続済み」と表示されたら完了

**トラブル時**は、マウスの充電確認やMacの再起動をお試しください。

ご不明な点やさらに詳しい説明が必要でしたら、お知らせください！

## クリーンアップ（オプション）

作成した AgentCore Runtime をクリーンアップしましょう

In [22]:
print(orchestrator_launch_result.ecr_uri, orchestrator_launch_result.agent_id, orchestrator_launch_result.ecr_uri.split('/')[1])
print(hr_launch_result.ecr_uri, hr_launch_result.agent_id, hr_launch_result.ecr_uri.split('/')[1])
print(tech_launch_result.ecr_uri, tech_launch_result.agent_id, tech_launch_result.ecr_uri.split('/')[1])

195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-orchestrator_agent orchestrator_agent-ZM5Zw2CRBP bedrock-agentcore-orchestrator_agent
195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-hr_agent hr_agent-N62XFxCDC5 bedrock-agentcore-hr_agent
195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-tech_agent tech_agent-uAy36F6CRS bedrock-agentcore-tech_agent


In [23]:
def clean_up_agent_runtimes(launch_result):
    agentcore_control_client = boto3.client(
        'bedrock-agentcore-control',
        region_name=os.getenv("AWS_DEFAULT_REGION")
    )
    ecr_client = boto3.client(
        'ecr',
        region_name=os.getenv("AWS_DEFAULT_REGION")
        
    )
    runtime_delete_response = agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )

    response = ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )

    return response

def delete_iam_roles(agentcore_iam_role):
    iam_client = boto3.client('iam')
    policies = iam_client.list_role_policies(
        RoleName=agentcore_iam_role['Role']['RoleName'],
        MaxItems=100
    )

    for policy_name in policies['PolicyNames']:
        iam_client.delete_role_policy(
            RoleName=agentcore_iam_role['Role']['RoleName'],
            PolicyName=policy_name
        )
    iam_response = iam_client.delete_role(
        RoleName=agentcore_iam_role['Role']['RoleName']
    )
    return iam_response

In [24]:
print(clean_up_agent_runtimes(hr_launch_result))
print(clean_up_agent_runtimes(tech_launch_result))
print(clean_up_agent_runtimes(orchestrator_launch_result))
print(delete_iam_roles(tech_agent_iam_role))
print(delete_iam_roles(hr_agent_iam_role))
print(delete_iam_roles(orchestrator_iam_role))

{'repository': {'repositoryArn': 'arn:aws:ecr:ap-northeast-1:195049633937:repository/bedrock-agentcore-hr_agent', 'registryId': '195049633937', 'repositoryName': 'bedrock-agentcore-hr_agent', 'repositoryUri': '195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-hr_agent', 'createdAt': datetime.datetime(2026, 1, 4, 15, 33, 1, 241000, tzinfo=tzlocal()), 'imageTagMutability': 'MUTABLE'}, 'ResponseMetadata': {'RequestId': '5b1bc487-db6e-4329-997b-2a59d4302d5b', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '5b1bc487-db6e-4329-997b-2a59d4302d5b', 'date': 'Sun, 04 Jan 2026 06:40:56 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '341', 'connection': 'keep-alive'}, 'RetryAttempts': 0}}
{'repository': {'repositoryArn': 'arn:aws:ecr:ap-northeast-1:195049633937:repository/bedrock-agentcore-tech_agent', 'registryId': '195049633937', 'repositoryName': 'bedrock-agentcore-tech_agent', 'repositoryUri': '195049633937.dkr.ecr.ap-northeast-1.amazonaws.c

# おめでとうございます！